# exp087_prefix_backtest_tvt_confidence train

Fold-safe prefix-backtest confidence calibration for PF/Beam/likelihood-PF TVT errors using the exp072 feature cache consumed by exp073.

## Contents

1. Setup and configuration
2. Source and schema check
3. Prefix-backtest confidence audit
4. Metrics and generated outputs

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from prefix_backtest_tvt_confidence import build_source_spec, run_audit, to_jsonable

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Anchor:", get_nested(config, "lineage.anchor"))
print("Cache parent:", get_nested(config, "lineage.cache_parent"))
print("Audit mode:", get_nested(config, "audit.mode"))
print("Debug:", DEBUG)

## 2. Source and schema check

In [ ]:
source = build_source_spec(config, paths.root)
print("Source name:", source.name)
print("Source path:", source.path)
print("Source exists:", source.path.exists())
print("Well column:", source.well_column)
print("Target column:", source.target_column)
print("Base column:", source.base_column)
print("X column:", source.x_column)
print("Primary candidate:", get_nested(config, "model.params.primary_candidate"))
print("Feature columns:")
for column in get_nested(config, "model.params.feature_columns"):
    print(" -", column)

if not source.path.exists():
    raise FileNotFoundError(source.path)

## 3. Prefix-backtest confidence audit

In [ ]:
summary = run_audit(config=config, paths=paths, debug=DEBUG)
print(json.dumps(to_jsonable(summary), indent=2, sort_keys=True))

## 4. Metrics and generated outputs

In [ ]:
output_paths = summary["outputs"]
for key, value in output_paths.items():
    print(f"{key}: {value}")

bin_metrics = pd.read_csv(output_paths["confidence_bin_metrics_csv"])
bucket_metrics = pd.read_csv(output_paths["bucket_metrics_csv"])
correlations = pd.read_csv(output_paths["signal_correlations_csv"])
display(bin_metrics)
display(bucket_metrics.head(20))
display(correlations.head(20))
print("metrics.json:", paths.metrics_path, paths.metrics_path.exists())